# RAG Retrieval Experiment: Dense vs BM25 vs Hybrid (Alpha Sweep)
Domain: synthetic Nginx admin knowledge base (60 docs, 180 test questions: 60 keyword / 60 semantic / 60 troubleshooting).

**Run every cell top to bottom in Google Colab.** Upload `docs.jsonl` and `qa_final_180.jsonl` to `/content/data/` first (Files panel on the left, or the upload cell below).

This notebook reports the *complete* alpha sweep (0.00 to 1.00) rather than cherry-picking the best-looking value, and computes MRR from the full ranked list rather than assuming MRR@1 == Recall@1.

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25

## Step 1: Upload data (or mount Drive)
Run this cell and pick `docs.jsonl` and `qa_final_180.jsonl` from your computer.

In [ ]:
from google.colab import files
import os
os.makedirs('/content/data', exist_ok=True)
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f'/content/data/{fname}')
print(os.listdir('/content/data'))

## Step 2: Load and validate data
Checks: every `expected_doc_id` actually exists in `docs.jsonl`, doc IDs are unique, and category counts match the 60/60/60 design.

In [ ]:
import json

docs = [json.loads(l) for l in open('/content/data/docs.jsonl')]
qa   = [json.loads(l) for l in open('/content/data/qa_final_180.jsonl')]

texts = [d['text'] for d in docs]
ids   = [d['id'] for d in docs]
id_set = set(ids)

assert len(ids) == len(id_set), "Duplicate doc IDs found!"

missing = [q for q in qa if q['expected_doc_id'] not in id_set]
assert not missing, f"{len(missing)} questions reference a doc_id that does not exist: {missing[:3]}"

from collections import Counter
cat_counts = Counter(q['category'] for q in qa)
print(f"{len(docs)} docs, {len(qa)} questions")
print("Category counts:", dict(cat_counts))
print("All expected_doc_id values validated against docs.jsonl: OK")

## Step 3: Build the three retrieval methods

- **Dense**: sentence-transformers `all-MiniLM-L6-v2` embeddings + FAISS `IndexFlatL2` (exact nearest neighbor, no approximation, so results are deterministic).
- **BM25**: `rank_bm25` classic keyword scoring over whitespace-tokenized, lowercased text.
- **Hybrid**: min-max normalize both score vectors independently, then combine as
  `hybrid_score = alpha * normalized_dense_similarity + (1 - alpha) * normalized_bm25_score`.
  Dense L2 distance is converted to a similarity (`-distance`) before normalizing, so higher is always better for both components.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=False)

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

def _norm(x):
    rng = x.max() - x.min()
    return (x - x.min()) / (rng + 1e-9)

def ranked_list_dense(query, k):
    q_emb = model.encode([query])
    dists, idx = index.search(q_emb, k)
    return [ids[i] for i in idx[0]]

def ranked_list_bm25(query, k):
    scores = np.array(bm25.get_scores(query.lower().split()))
    top = np.argsort(scores)[::-1][:k]
    return [ids[i] for i in top]

def ranked_list_hybrid(query, k, alpha):
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    q_emb = model.encode([query])
    vec_scores = -np.linalg.norm(doc_embeddings - q_emb, axis=1)  # higher = closer
    combined = alpha * _norm(vec_scores) + (1 - alpha) * _norm(bm25_scores)
    top = np.argsort(combined)[::-1][:k]
    return [ids[i] for i in top]

## Step 4: Evaluation metrics — careful MRR implementation

**Important:** MRR is computed from a *single full ranked list per query* (we retrieve k=5, rank the full top-5, and score `1/rank` if the correct doc appears, else 0), NOT recomputed separately per k. This means MRR is a single fixed number for a given method, and is deliberately **not** forced to equal Recall@1. Recall@k for k in {1,3,5} is a separate, simpler metric: "was the correct doc anywhere in the top k".

In [ ]:
def evaluate(rank_fn, k_max=5):
    """rank_fn(query, k) -> ranked list of doc ids, length k.
    Returns overall + per-category Recall@1/3/5 and MRR (from the full top-5 ranking)."""
    per_cat = {c: {"n": 0, "r1": 0, "r3": 0, "r5": 0, "rr_sum": 0.0} for c in ["keyword", "semantic", "troubleshooting"]}
    overall = {"n": 0, "r1": 0, "r3": 0, "r5": 0, "rr_sum": 0.0}
    fails_at_1, fails_at_3 = [], []

    for item in qa:
        full_rank = rank_fn(item['question'], k_max)  # single ranked list, length k_max
        target = item['expected_doc_id']
        cat = item['category']

        hit1 = target in full_rank[:1]
        hit3 = target in full_rank[:3]
        hit5 = target in full_rank[:5]
        rr = 0.0
        if target in full_rank:
            rr = 1.0 / (full_rank.index(target) + 1)

        for bucket in (overall, per_cat[cat]):
            bucket["n"] += 1
            bucket["r1"] += int(hit1)
            bucket["r3"] += int(hit3)
            bucket["r5"] += int(hit5)
            bucket["rr_sum"] += rr

        if not hit1:
            fails_at_1.append({"id": item['id'], "question": item['question'], "category": cat,
                                "expected_doc_id": target, "retrieved": full_rank})
        if not hit3:
            fails_at_3.append({"id": item['id'], "question": item['question'], "category": cat,
                                "expected_doc_id": target, "retrieved": full_rank})

    def summarize(b):
        return {"recall@1": b["r1"]/b["n"], "recall@3": b["r3"]/b["n"],
                "recall@5": b["r5"]/b["n"], "mrr": b["rr_sum"]/b["n"], "n": b["n"]}

    return {
        "overall": summarize(overall),
        "by_category": {c: summarize(b) for c, b in per_cat.items()},
        "fails_at_1": fails_at_1,
        "fails_at_3": fails_at_3,
    }

## Step 5: Run Dense-only and BM25-only

In [ ]:
dense_results = evaluate(lambda q, k: ranked_list_dense(q, k))
bm25_results   = evaluate(lambda q, k: ranked_list_bm25(q, k))

print("DENSE ONLY:", json.dumps(dense_results['overall'], indent=2))
print("BM25 ONLY: ", json.dumps(bm25_results['overall'], indent=2))

## Step 6: Full alpha sweep (0.00 to 1.00) — do NOT skip to just the "best" value

alpha=1.00 is mathematically equivalent to dense-only; alpha=0.00 is equivalent to BM25-only (both are included in the table for a direct sanity check against Step 5).

In [ ]:
alpha_values = [0.00, 0.25, 0.50, 0.75, 1.00]
alpha_sweep = {}
for a in alpha_values:
    res = evaluate(lambda q, k, a=a: ranked_list_hybrid(q, k, a))
    alpha_sweep[a] = res
    o = res['overall']
    print(f"alpha={a:.2f}  Recall@1={o['recall@1']:.4f}  Recall@3={o['recall@3']:.4f}  "
          f"Recall@5={o['recall@5']:.4f}  MRR={o['mrr']:.4f}")

In [ ]:
import pandas as pd

sweep_table = pd.DataFrame([
    {"alpha": a, "Recall@1": r['overall']['recall@1'], "Recall@3": r['overall']['recall@3'],
     "Recall@5": r['overall']['recall@5'], "MRR": r['overall']['mrr']}
    for a, r in alpha_sweep.items()
])
sweep_table

## Step 7: Category breakdown (keyword / semantic / troubleshooting) for each method

In [ ]:
def category_table(results_dict, label):
    rows = []
    for method, res in results_dict.items():
        for cat, m in res['by_category'].items():
            rows.append({"method": method, "category": cat, "Recall@1": m['recall@1'],
                         "Recall@3": m['recall@3'], "Recall@5": m['recall@5'], "MRR": m['mrr'], "n": m['n']})
    return pd.DataFrame(rows)

methods_for_category_table = {
    "dense_only": dense_results,
    "bm25_only": bm25_results,
    **{f"hybrid_alpha_{a}": r for a, r in alpha_sweep.items()},
}
cat_df = category_table(methods_for_category_table, "by category")
cat_df

## Step 8: Error analysis for the best-performing configuration

Pick the config with the highest overall Recall@1 from the sweep (ties broken by MRR) — this is chosen *after* seeing the full table above, not assumed in advance.

In [ ]:
all_configs = {"dense_only": dense_results, "bm25_only": bm25_results,
               **{f"hybrid_alpha_{a}": r for a, r in alpha_sweep.items()}}

best_name = max(all_configs, key=lambda k: (all_configs[k]['overall']['recall@1'], all_configs[k]['overall']['mrr']))
best = all_configs[best_name]
print(f"Best config by Recall@1 (tie-break MRR): {best_name}")
print(json.dumps(best['overall'], indent=2))

doc_topic = {d['id']: d.get('topic', '') for d in docs}

fail_rows = []
for f in best['fails_at_3']:
    fail_rows.append({
        "question": f['question'],
        "category": f['category'],
        "expected_doc": f"{f['expected_doc_id']} ({doc_topic.get(f['expected_doc_id'],'')})",
        "retrieved_docs": f['retrieved'],
        "retrieved_topics": [doc_topic.get(d, '') for d in f['retrieved']],
    })

fail_df = pd.DataFrame(fail_rows)
print(f"\nFailed at Recall@1: {len(best['fails_at_1'])} / {best['overall']['n']} queries")
print(f"Failed at Recall@3: {len(best['fails_at_3'])} / {best['overall']['n']} queries")
fail_df

## Step 9: Save everything needed for the paper

In [ ]:
final_export = {
    "dense_only": dense_results['overall'],
    "bm25_only": bm25_results['overall'],
    "alpha_sweep_overall": {str(a): r['overall'] for a, r in alpha_sweep.items()},
    "alpha_sweep_by_category": {str(a): r['by_category'] for a, r in alpha_sweep.items()},
    "dense_by_category": dense_results['by_category'],
    "bm25_by_category": bm25_results['by_category'],
    "best_config": best_name,
    "best_config_metrics": best['overall'],
    "fails_at_1_count": len(best['fails_at_1']),
    "fails_at_3_count": len(best['fails_at_3']),
}
with open('/content/final_results.json', 'w') as f:
    json.dump(final_export, f, indent=2)

cat_df.to_csv('/content/category_breakdown.csv', index=False)
fail_df.to_csv('/content/error_analysis.csv', index=False)
sweep_table.to_csv('/content/alpha_sweep.csv', index=False)

print("Saved: final_results.json, category_breakdown.csv, error_analysis.csv, alpha_sweep.csv")
print("Download these four files (Files panel, left sidebar) and send them back for the paper.")

## What to actually report in the paper
- The **alpha_sweep table** (Step 6) in full — including alpha values where hybrid does *not* beat dense-only, if that happens. A flat or non-monotonic curve is a real, reportable finding.
- The **category breakdown** (Step 7) — this is the strongest evidence either way: does hybrid's advantage (if any) concentrate in the `keyword` category, as the original hypothesis predicted, or is it flat across categories?
- The **error analysis** (Step 8) — a handful of concrete failed-query examples, with what was retrieved instead, makes the paper's Results/Discussion section much stronger than aggregate numbers alone.
- Report `fails_at_1_count` / `fails_at_3_count` out of 180 as plain failure counts.
- If dense-only turns out to beat hybrid at every alpha, report that as the actual finding — it is still a valid, publishable result for a short paper, and is scientifically more honest than only reporting a cherry-picked alpha.